- Purpose:Ingestion of Data
- Author: Bledi Rexha
- Last updated: 06/10/2026  
- Dependencies:
Delta Live Tables (DLT),
Auto Loader,
PySpark,
Unity Catalog,
Azure Data Lake Storage Gen2

In [0]:
import dlt
import pandas as pd
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit

# SAS URI is read from the DLT pipeline configuration
#   key:   capstone.metadata_sas_uri
#   value: https://capstonemetadata.blob.core.windows.net/sources/metadata.csv?sp=r&...
SAS_URI = spark.conf.get("capstone.metadata_sas_uri", None)
if not SAS_URI:
    raise ValueError("Missing pipeline config: capstone.metadata_sas_uri")

# 26 columns in source order.
metadata_schema = StructType([
    StructField("column_id", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("column_desc", StringType(), True),
    StructField("term_name", StringType(), True),
    StructField("term_description", StringType(), True),
    StructField("security_classification", StringType(), True),
    StructField("critical_data_element_flag", StringType(), True),
    StructField("pii_flag", StringType(), True),
    StructField("term_subdomain", StringType(), True),
    StructField("data_steward", StringType(), True),
    StructField("table_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("table_desc", StringType(), True),
    StructField("table_owner_in_source", StringType(), True),
    StructField("schema_id", StringType(), True),
    StructField("schema_name", StringType(), True),
    StructField("database_id", StringType(), True),
    StructField("database_name", StringType(), True),
    StructField("system_id", StringType(), True),
    StructField("system_name", StringType(), True),
    StructField("location", StringType(), True),
    StructField("total_record_count", StringType(), True),
    StructField("invalid_record_count", StringType(), True),
    StructField("tag_name", StringType(), True),
    StructField("tag_value", StringType(), True),
    StructField("certification_level", StringType(), True),
])


@dlt.table(
    name="metadata_raw_bronze",
    comment="Raw metadata ingested from Azure Blob Storage into the Bronze layer"
)
def metadata_raw_bronze():
    pdf = pd.read_csv(SAS_URI, dtype=str)
    expected_columns = [field.name for field in metadata_schema.fields]
    pdf = pdf[expected_columns]
    df = spark.createDataFrame(pdf, schema=metadata_schema)
    return (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_file", lit(SAS_URI.split("?")[0]))
    )